# Lab Day 2 — 2A202602810 Nguyễn Thái Anh

Triển khai đến hết **mục 4** của hướng dẫn: setup + dataset/model/loss/train. Các phần EDA đầy đủ, thí nghiệm B/T/I/F và sản phẩm cuối chưa chạy. Các cell setup không tự mở test.


## 1. Đường dẫn và môi trường

Chọn kernel .venv/bin/python trong IDE; trên Colab/Kaggle cài code/requirements.txt bằng phiên bản torch/torchvision phù hợp runtime. Sửa ROOT nếu cwd không phải gốc repo.


In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()  # chỉnh thành gốc repo nếu cần
if not (ROOT / "eval.py").is_file():
    raise FileNotFoundError("Đặt ROOT tới gốc repo có eval.py")
SUBMISSION = ROOT / "submissions/2A202602810_NguyenThaiAnh"
CODE = SUBMISSION / "code"
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))

import torch, torchvision, timm
import dataset, model, losses, train
from train import Config, run
print("train:", train.__file__)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__, "timm:", timm.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 2. CSV nguyên bản và đường dẫn ảnh

Ảnh local nằm trực tiếp trong data/. Script chỉ tải CSV, không tải lại ảnh. Header split gốc là Filename,Label; Species được thêm trong bộ nhớ từ mapping lớp. Nếu đã có đủ CSV và provenance, setup hoạt động offline.


In [ ]:
IMAGES_DIR = ROOT / "data"
LABELS_DIR = ROOT / "data/labels"
required = ["labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv"]
if not all((LABELS_DIR / n).is_file() for n in required):
    from setup_data import download_labels
    download_labels(LABELS_DIR, CODE / "data_sources.json")
if not any(IMAGES_DIR.glob("*.jpg")):
    raise FileNotFoundError("Tải/giải nén ảnh và chỉnh IMAGES_DIR trước khi tiếp tục")
print("images:", IMAGES_DIR)
print("labels:", LABELS_DIR)


## 3. Audit setup

Đây là kiểm tra metadata/file, không forward test qua model. Audit full decode đã lưu trong code/validation/split_check.json. Cell này ghi audit nhanh vào runs/setup_audit để giữ nguyên bằng chứng full decode. Ghi nhận chênh lệch nhãn nguồn và giữ split nguyên bản.


In [ ]:
from validate_setup import validate_setup
import pandas as pd

report = validate_setup(str(IMAGES_DIR), str(LABELS_DIR), ROOT / "runs/setup_audit", decode_images=False)
counts = pd.DataFrame(report["per_class"])
counts.index = dataset.CLASS_NAMES
print(counts)


## 4. Kiểm tra code bài làm

Chạy độc lập với suite tests/ gốc để không import nhầm starter. Fixtures tổng hợp dùng kiểm tra phần mềm, không phải kết quả DeepWeeds.


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(CODE / "tests"), "-v"], cwd=ROOT, check=True)


## 5. Config chuẩn bị cho giai đoạn tiếp theo

Cell dưới chỉ dựng config, **không train**. Trước khi gọi run(cfg), hoàn thành EDA, kiểm tra ảnh augmentation, CE ban đầu và overfit một batch như mục 5 của hướng dẫn.


In [ ]:
cfg = Config(
    exp_id="B01", seed=0, backbone="resnet50",
    images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
    out_dir=str(ROOT / "runs"), pred_dir=str(SUBMISSION / "predictions"),
    curves_dir=str(SUBMISSION / "curves"),
    save_test_predictions=False,
)
train.validate_config(cfg)
print(cfg)


## Các giai đoạn còn lại

EDA/sanity → pilot GPU → ≥5 backbone → ≥3 trục training + kết hợp → I00 + ≥4 phương pháp inference/latency → khóa cấu hình trên val → final và baseline ≥3 seed → test một lần/seed → evaluator → Excel/report.

inference.py và benchmark.py hiện còn là khung. Chỉ bật nhánh test sau khi nối đúng phương pháp cuối đã khóa vào final evaluator.
